In [ ]:
!pip install catboost

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math as mt
import lightgbm as lgb
import shap
import json
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, StratifiedKFold, TimeSeriesSplit
from catboost import CatBoostClassifier, Pool

# Чтение данных

In [ ]:
import os

DATA_PATH = "data/processed"  # обработанные данные
MODELS_PATH = "models/logreg"  # сохранение модели

os.makedirs(MODELS_PATH, exist_ok=True)

In [ ]:
RANDOM_SEED = 91

In [ ]:
np.random.seed(RANDOM_SEED)

In [ ]:
df = pd.read_csv(DATA_PATH + "/uplift-dataset.csv")

print("Dataset shape:", df.shape)

Dataset shape: (307511, 161)


# Посмотрим на данные

In [ ]:
df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,SK_DPD_MAX_CC,BASE_PD,CONTACT_PROPENSITY,RISK_SEGMENT,CONTACT_HISTORY,PREFERRED_CHANNEL,INTERACTION_SCORE,DELAY_FLAG,COMMUNICATION,TRUE_UPLIFT
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,NaN,0.950968,0.609805,high_risk,9,sms,0.450668,1,operator_call,0.028513
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,NaN,0.608977,0.456933,high_risk,0,operator_call,0.341462,1,control,-0.005344
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,NaN,0.227401,0.148340,low_risk,0,sms,0.483622,0,control,0.003340
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.0,0.359173,0.253956,low_risk,2,robot_call,0.461974,0,control,-0.021130
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,NaN,0.629868,0.420203,high_risk,2,robot_call,0.437186,0,control,-0.004420
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,NaN,0.661227,0.430674,high_risk,3,operator_call,0.469229,1,control,-0.001041
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,NaN,0.671782,0.425683,high_risk,2,sms,0.467030,0,operator_call,-0.068337
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,NaN,0.423995,0.311099,medium_risk,0,sms,0.416992,0,control,-0.010546
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,NaN,0.376648,0.259927,low_risk,1,sms,0.455039,1,control,-0.012077


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 161 entries, SK_ID_CURR to TRUE_UPLIFT
dtypes: float64(99), int64(43), object(19)
memory usage: 377.7+ MB


In [ ]:
df['TARGET'].value_counts(normalize=True).round(4) * 100

,proportion
TARGET,
0,91.93
1,8.07


# Посмотрим на переменные

In [ ]:
dt_cols = df.select_dtypes(include=['datetime64']).columns.to_list()
object_cols = df.select_dtypes(include=['object']).columns.to_list()
num_cols = df.select_dtypes(include=['number']).columns.to_list()
flg_cols = [col for col in df.columns if col.startswith("FLAG_")] + [
    "LIVE_CITY_NOT_WORK_CITY",
    "LIVE_REGION_NOT_WORK_REGION",
    "REG_CITY_NOT_LIVE_CITY",
    "REG_CITY_NOT_WORK_CITY",
    "REG_REGION_NOT_LIVE_REGION",
    "REG_REGION_NOT_WORK_REGION"
]


target = "TARGET"

uplift_cols = [
  "BASE_PD",
  "CONTACT_PROPENSITY",
  "COMMUNICATION",
  "RISK_SEGMENT",
  "CONTACT_HISTORY",
  "PREFERRED_CHANNEL",
  "INTERACTION_SCORE",
  "DELAY_FLAG",
  "TRUE_UPLIFT",
]


num_cols = sorted(list(set(num_cols) - set(flg_cols)))

num_cols.remove('TARGET')

In [ ]:
dt_cols

[]

In [ ]:
object_cols

['NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'WEEKDAY_APPR_PROCESS_START',
 'ORGANIZATION_TYPE',
 'FONDKAPREMONT_MODE',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE',
 'EMERGENCYSTATE_MODE',
 'RISK_SEGMENT',
 'PREFERRED_CHANNEL',
 'COMMUNICATION']

In [ ]:
num_cols

['AMT_ANNUITY',
 'AMT_APPLICATION_MAX',
 'AMT_APPLICATION_MEAN',
 'AMT_BALANCE_MAX',
 'AMT_BALANCE_MEAN',
 'AMT_CREDIT',
 'AMT_CREDIT_LIMIT_ACTUAL_MEAN',
 'AMT_CREDIT_MAX',
 'AMT_CREDIT_MEAN',
 'AMT_CREDIT_SUM_DEBT_MEAN',
 'AMT_CREDIT_SUM_DEBT_SUM',
 'AMT_CREDIT_SUM_MEAN',
 'AMT_CREDIT_SUM_OVERDUE_SUM',
 'AMT_CREDIT_SUM_SUM',
 'AMT_DRAWINGS_CURRENT_MEAN',
 'AMT_GOODS_PRICE',
 'AMT_INCOME_TOTAL',
 'AMT_INSTALMENT_MEAN',
 'AMT_PAYMENT_MEAN',
 'AMT_PAYMENT_SUM',
 'AMT_REQ_CREDIT_BUREAU_DAY',
 'AMT_REQ_CREDIT_BUREAU_HOUR',
 'AMT_REQ_CREDIT_BUREAU_MON',
 'AMT_REQ_CREDIT_BUREAU_QRT',
 'AMT_REQ_CREDIT_BUREAU_WEEK',
 'AMT_REQ_CREDIT_BUREAU_YEAR',
 'APARTMENTS_AVG',
 'APARTMENTS_MEDI',
 'APARTMENTS_MODE',
 'BASEMENTAREA_AVG',
 'BASEMENTAREA_MEDI',
 'BASEMENTAREA_MODE',
 'BASE_PD',
 'CNT_CHILDREN',
 'CNT_FAM_MEMBERS',
 'CNT_INSTALMENT_FUTURE_MEAN',
 'CNT_INSTALMENT_MEAN',
 'CNT_PAYMENT_MEAN',
 'COMMONAREA_AVG',
 'COMMONAREA_MEDI',
 'COMMONAREA_MODE',
 'CONTACT_HISTORY',
 'CONTACT_PROPENSITY',
 '

In [ ]:
flg_cols

['FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'FLAG_DOCUMENT_2',
 'FLAG_DOCUMENT_3',
 'FLAG_DOCUMENT_4',
 'FLAG_DOCUMENT_5',
 'FLAG_DOCUMENT_6',
 'FLAG_DOCUMENT_7',
 'FLAG_DOCUMENT_8',
 'FLAG_DOCUMENT_9',
 'FLAG_DOCUMENT_10',
 'FLAG_DOCUMENT_11',
 'FLAG_DOCUMENT_12',
 'FLAG_DOCUMENT_13',
 'FLAG_DOCUMENT_14',
 'FLAG_DOCUMENT_15',
 'FLAG_DOCUMENT_16',
 'FLAG_DOCUMENT_17',
 'FLAG_DOCUMENT_18',
 'FLAG_DOCUMENT_19',
 'FLAG_DOCUMENT_20',
 'FLAG_DOCUMENT_21',
 'LIVE_CITY_NOT_WORK_CITY',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION']

# Сгенерируем новые перменные

In [ ]:
# Возраст в годах
df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365

# Стаж работы
df["EMPLOYMENT_YEARS"] = -df["DAYS_EMPLOYED"] / 365

# Доля стажа от возраста
df["EMPLOYMENT_RATIO"] = df["EMPLOYMENT_YEARS"] / df["AGE_YEARS"]

# Давность кредитной истории
df["CREDIT_HISTORY_LENGTH"] = (
    df["DAYS_CREDIT_MAX"] - df["DAYS_CREDIT_MIN"]
)

# Активность по кредитке
df["MONTHS_BALANCE_RANGE"] = (
    df["MONTHS_BALANCE_MIN"] - df["MONTHS_BALANCE_MEAN"]
)

# Обработаем датасет

In [ ]:
def transform_to_standart_types(df: pd.DataFrame) -> pd.DataFrame:
    """
    Преобразует все колонки к стандартным типам данных

    Args:
        df (pd.DataFrame): Датафрейм с данными для обучения модели
    Returns:
        df (pd.DataFrame): Датафрейм с данными для обучения модели с преобразованными колонками
    """

    cat_features = df.select_dtypes(object).columns.to_list()

    df[cat_features] = df[cat_features].replace(np.nan, 'NaN')

    for feature in cat_features:
        df[feature] = df[feature].astype(str)

    for col in df.select_dtypes(include=[np.int64]).columns.to_list():
        df[col] = df[col].astype(np.float64)

    return df

def fillna_categorical(df: pd.DataFrame) -> pd.DataFrame:
    """
    Заполняет пропуски в категориальных переменных

    Args:
        df (pd.DataFrame): Датафрейм с данными для обучения модели
    Returns:
        df (pd.DataFrame): Датафрейм с данными для обучения модели с преобразованными колонками
    """

    cat_features = df.select_dtypes(object).columns.to_list()

    df[cat_features] = df[cat_features].fillna('NaN')
    df[cat_features] = df[cat_features].replace([np.nan, 'None', 'N/A'], 'NaN')

    return df

def drop_unnecessary_columns(df: pd.DataFrame, max_missing_ratio=0.8) -> pd.DataFrame:
    """
    Удаляет ненужные, константные колонки, а также колонки с большой долей пропусков

    Args:
        df (pd.DataFrame): Датафрейм с данными для обучения модели
        max_missing_ratio (float): Порог отсечения переменных с большой долей пропусков
    Returns:
        df (pd.DataFrame): Датафрейм с данными для обучения модели c удаленными ненужными колонками
    """

    unnecessary_columns = ['SK_ID_CURR']

    for col in unnecessary_columns:
        if col in df.columns:
            df = df.drop(col, axis=1)

    missing_ratio = df.isna().mean()
    high_missing_ratio = missing_ratio[missing_ratio > max_missing_ratio].index.to_list()
    df = df.drop(high_missing_ratio, axis=1)

    constant_columns = df.nunique(dropna=False).to_frame()
    constant_columns = constant_columns[constant_columns[0] < 2].index.to_list()
    df = df.drop(constant_columns, axis=1)

    return df

In [ ]:
X = df.drop(columns=target).copy()
y = df[target].copy()

In [ ]:
X = fillna_categorical(X)
X = transform_to_standart_types(X)

In [ ]:
X = drop_unnecessary_columns(X)

In [ ]:
filtred_cols = X.columns.tolist()

Посмотрим нет ли признаков, слишком коррелирующих с таргетом

In [ ]:
corr = df[num_cols].corrwith(df["TARGET"]).abs().sort_values(ascending=False).round(2)

In [ ]:
corr

,0
BASE_PD,0.20
CONTACT_PROPENSITY,0.18
EXT_SOURCE_3,0.18
EXT_SOURCE_2,0.16
EXT_SOURCE_1,0.16
...,...
AMT_REQ_CREDIT_BUREAU_QRT,0.00
NONLIVINGAPARTMENTS_MODE,0.00
AMT_REQ_CREDIT_BUREAU_HOUR,0.00
AMT_REQ_CREDIT_BUREAU_WEEK,0.00


In [ ]:
print(f'Из исходных {len(df.columns.tolist())} переменных для построения будем рассматривать {len(filtred_cols)}')

Из исходных 166 переменных для построения будем рассматривать 164


# Разделим данные на выборки

In [ ]:
data = X.copy()
data[target] = y.copy()

In [ ]:
train_test  = data[mt.ceil(len(data)*0.2):].copy(deep=True)
oot  = data[:mt.ceil(len(data)*0.2)].copy(deep=True)

X_train, X_test, y_train, y_test = train_test_split(train_test.drop(columns=target), train_test[target],
                                                    test_size=0.25, stratify=train_test[target])

X_oot, y_oot = oot.drop(columns=target), oot[target]

train = X_train.copy()
train[target] = y_train

test = X_test.copy()
test[target] = y_test

In [ ]:
print(f'Доля наблюдений на train: {len(X_train)/len(data) * 100:.2f}%')
print(f'Доля наблюдений на test: {len(X_test)/len(data) * 100:.2f}%')
print(f'Доля наблюдений на oot: {len(X_oot)/len(data) * 100:.2f}%')

Доля наблюдений на train: 60.00%
Доля наблюдений на test: 20.00%
Доля наблюдений на oot: 20.00%


In [ ]:
print(f'Общее количество наблюдений: {len(data)}')
print(f'Количество наблюдений на train: {len(X_train)}')
print(f'Количество наблюдений на test: {len(X_test)}')
print(f'Количество наблюдений на oot: {len(X_oot)}')

Общее количество наблюдений: 307511
Количество наблюдений на train: 184506
Количество наблюдений на test: 61502
Количество наблюдений на oot: 61503


In [ ]:
print(f'Общий BR на всех данных: {round(data[target].mean() * 100, 4)}%')
print(f'BR на train: {round(y_train.mean() * 100, 4)}%')
print(f'BR на test: {round(y_test.mean() * 100, 4)}%')
print(f'BR на oot: {round(y_oot.mean() * 100, 4)}%')

Общий BR на всех данных: 8.0729%
BR на train: 8.0881%
BR на test: 8.0875%
BR на oot: 8.0126%


# Произведем отбор переменных для построения модели

In [ ]:
features = data.columns.tolist()
features.remove('TARGET')

In [ ]:
features = [col for col in features if col not in uplift_cols]

## 1. Посчитаем IV и Gini по всем переменным, затем отфильтруем по корреляции

In [ ]:
def get_iv(df, feature, y, bins=None):
    """
    Возвращает значение IV
    Args:
    df (pd.DataFrame): Датафрейм, на котором будет подсчитываться IV
    feature (str): признак для подсчета IV
    y (pd.Series | np.ndarray): Истинные значения
    bins (int | None): число бинов для разбиения
    Returns:
    Значение IV
    """
    lst = []
    target = 'y'
    df[target] = y
    if bins:
        df[feature] = pd.cut(df[feature], bins=bins)

    unique_values = df[feature].unique()
    for val in unique_values:
        lst.append([feature,                                                        # Feature name
                    val,                                                            # Value of a feature (unique)
                    df[(df[feature] == val) & (df[target] == 0)].count()[feature],  # Good (Fraud == 0)
                    df[(df[feature] == val) & (df[target] == 1)].count()[feature]   # Bad  (Fraud == 1)
                    ])

    data = pd.DataFrame(lst, columns=['Variable', 'Value', 'Good', 'Bad'])

    total_bad = df[df[target] == 1].count()[feature]
    total_good = df.shape[0] - total_bad

    data['Distribution Good'] = data['Good']/ total_good
    data['Distribution Bad'] = data['Bad'] / total_bad
    data['WoE'] = np.log(data['Distribution Good'] / data['Distribution Bad'])

    data = data.replace({'WoE': {np.inf: 0, -np.inf: 0}})

    data['IV'] = data['WoE'] * (data['Distribution Good'] - data['Distribution Bad'])

    data = data.sort_values(by=['Variable', 'Value'], ascending=[True, True])
    data.index = range(len(data.index))

    iv = data['IV'].sum()

    return iv

In [ ]:
iv_df = list()

for col in tqdm(features):
    if col not in cat_features:
        iv = get_IV(train.copy(), col, target, bins=10)
    else:        iv = get_IV(train.copy(), col, target)
    iv_df.append({"var": col, "iv": iv})

iv_df = pd.DataFrame(iv_df)

filtred_features = iv_df[iv_df["iv"] > 0.005]["var"].values.tolist()

filtred_iv_report = iv_df.rename(columns={'var': 'Name', 'iv': 'IV'})

features = sorted(filtred_features)

In [ ]:
len(features)

In [ ]:
features_gini = pd.DataFrame(columns=['Name', 'Value'])

# Для каждой переменной будем производить кодирование по WoE и вычислять соло-Джини
for feature in tqdm(features):
    binning_process = BinningProcess(
        categorical_variables=cat_features,
        variable_names=[feature]
    )

    binning_process.fit(pd.DataFrame(X_train[feature]), y_train)
    X_train_binned = binning_process.transform(pd.DataFrame(X_train[feature]), metric="woe")
    X_test_binned = binning_process.transform(pd.DataFrame(X_test[feature]), metric="woe")

    # Обучим модель    model = LR(random_state=RANDOM_SEED).fit(X_train_binned, y_train)

    # Посчитаем Джини
    row = {'Name': feature, 'Value': get_gini(y_test, (model.predict_proba(X_test_binned)[:, 1]).round(2))}
    features_gini = pd.concat([features_gini, pd.DataFrame([row])], ignore_index=True)

features_gini = features_gini.sort_values(by=['Value', 'Name'], ascending=False)
filtred_iv_report = filtred_iv_report.sort_values(by=['IV', 'Name'], ascending=False)

In [ ]:
features_gini

In [ ]:
filtred_iv_report

In [ ]:
features_gini = features_gini[features_gini['Name'].isin(filtred_features)]
filtred_features = features_gini[features_gini['Value'] > 0]['Name'].values.tolist()

In [ ]:
len(filtred_features)

## 2. Закодируем переменные по WoE и отфильтруем по корреляции с порогом 0.35

In [ ]:
features = filtred_features

In [ ]:
binning_process = BinningProcess(
    categorical_variables=cat_features,    variable_names=features,)

binning_process.fit(X_train[features], y_train)

In [ ]:
X_train_binned = binning_process.transform(X_train, metric="woe")
X_test_binned = binning_process.transform(X_test, metric="woe")
X_oot_binned = binning_process.transform(X_oot, metric="woe")

In [ ]:
train_binned = X_train_binned
train_binned[target] = y_train

test_binned = X_test_binned
test_binned[target] = y_test

oot_binned = X_oot_binned
oot_binned[target] = y_oot

### Отфтильтруем по корреляции с порогом 0.35

In [ ]:
corr_matrix = X_train_binned[features].corr().abs()
to_drop = set()

thresold = 0.35

checked_cols = []

for i in tqdm(range(len(features))):
    for j in range(i + 1, len(features)):
        col1 = features[i]
        col2 = features[j]
        if col2 != col1 and corr_matrix[col1][col2] >= thresold:
            if filtred_iv_report[filtred_iv_report['Name']==col1]['IV'].values < filtred_iv_report[filtred_iv_report['Name']==col2]['IV'].values:
                to_drop.add(col1)
            else:
                to_drop.add(col2)

In [ ]:
len(to_drop)

In [ ]:
filtred_corr = filtred_iv_report[filtred_iv_report['Name'].isin(filtred_features)].sort_values(by='IV', ascending=False)['Name'].values.tolist()

In [ ]:
filtred_corr = [col for col in filtred_corr if col not in to_drop]

In [ ]:
len(filtred_corr)

## Применим метод отбора переменных Stepwise

In [ ]:
filtred_report_best = filtred_iv_report[filtred_iv_report['Name'].isin(filtred_corr)].merge(features_gini, on='Name')

In [ ]:
filtred_report_best = filtred_report_best.rename(columns={'Value': 'Gini'})

In [ ]:
filtred_report_best = filtred_report_best[filtred_report_best['Gini'] > 5]

In [ ]:
filtred_report_best = filtred_report_best[filtred_report_best['IV'] > 0.05]

In [ ]:
filtred_report_best = filtred_report_best.sort_values(by=['Gini'], ascending=False)

In [ ]:
filtred_report_best

In [ ]:
features = filtred_report_best['Name'].values.tolist()

In [ ]:
# Реализация кроссвалидации с разбиением переменных по WOE. Отличие только в использовании tqdm для вывода информации об итерациях
def my_binnig_tqdm_cv(X, y, cv):
    score = []
    for i, (train_index, test_index) in enumerate(tqdm(cv.split(X, y), total=cv.get_n_splits(), desc='CV')):
        # Закодируем переменные по WOE
        cat_features = X_train.select_dtypes(include=['category','object']).columns.to_list()
        features = X.columns.to_list()

        binning_process = BinningProcess(
            categorical_variables=cat_features,            variable_names=features,        )

        binning_process.fit(X.iloc[train_index], y.iloc[train_index])

        X_binned = binning_process.transform(X, metric="woe")

        # Обучим модель        model = LR(random_state=RANDOM_SEED).fit(X_binned.iloc[train_index], y.iloc[train_index])

        gini_train = get_gini(y.iloc[train_index], model.predict_proba(X_binned.iloc[train_index])[:, 1])
        gini_test = get_gini(y.iloc[test_index], model.predict_proba(X_binned.iloc[test_index])[:, 1])

        # Сохраним показатели Джини для каждого фолда
        score.append({
            "train": gini_train,
            "test": gini_test,
         })

        tqdm.write(f'Fold {i+1}, gini_train: {gini_train:.2f}, gini_test: {gini_test:.2f}, train_size: {len(X_binned.iloc[train_index])},\
                       test_size: {len(X_binned.iloc[test_index])}, br_train: {y.iloc[train_index].mean():.2f}, br_test: {y.iloc[test_index].mean():.2f}')
    return pd.DataFrame(score).mean()

In [ ]:
# Разбиение на фолды с учётом временного ряда
tscv = TimeSeriesSplit()

In [ ]:
# Будем включать в обучение выборки по одной переменной из списка и оценивать качество модели на кроссвалидации
res = []
for i in range(1, len(features) + 1):
    ans = my_binnig_cv(X_train[features[:i]], y_train, tscv)
    res.append({"top": i, "train_cv_score": ans['train'], 'test_cv_score': ans['test']})
    print(f"top: {i}, train_cv_score: {ans['train']}, test_cv_score: {ans['test']}")

In [ ]:
# Построим график распределения качества модели в зависимости от количества переменных
res = pd.DataFrame(res)
res = res.sort_values(by='top')
res.plot(x='top', y=['train_cv_score', 'test_cv_score'],
       figsize=(8,4),
)
plt.show()

## Посмотрим на отобранные переменные

In [ ]:
filtred_features = filtred_report_best['Name'].head(15).values.tolist()

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='IV', y='Name', data=filtred_iv_report[filtred_iv_report['Name'].isin(filtred_features)], palette="Blues_d")
plt.title('Значение IV по переменным')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Value', y='Name', data=features_gini[features_gini['Name'].isin(filtred_features)], palette="Blues_d")
plt.title('Значение Джини по переменным')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(17,17))
sns.heatmap(X_train_binned[filtred_features].corr().abs(), annot=True, ax=ax,  fmt=".2")
plt.show()

# Обучим модель на отобранных переменных

In [ ]:
cat_features = X_train[filtred_features].select_dtypes(include=['object', 'category']).columns.to_list()

In [ ]:
binning_process = BinningProcess(
    categorical_variables=cat_features,    variable_names=filtred_features,)

binning_process.fit(X_train[filtred_features], y_train)

In [ ]:
X_train_binned = binning_process.transform(X_train, metric="woe")
X_test_binned = binning_process.transform(X_test, metric="woe")
X_oot_binned = binning_process.transform(X_oot, metric="woe")
X_prom_oot_binned = binning_process.transform(prom_oot[filtred_features], metric="woe")

In [ ]:
train_binned = X_train_binned
train_binned[target] = y_train

test_binned = X_test_binned
test_binned[target] = y_test

oot_binned = X_oot_binned
oot_binned[target] = y_oot

In [ ]:
m = LR(random_state=RANDOM_SEED).fit(X_train_binned[filtred_features], y_train)

In [ ]:
res = get_gini_by_segments(filtred_features, m, target,
                          train_binned, test_binned, oot_binned, prom_oot_binned
                          )
plot_results(res)

# Подберем гиперпараметры

## Подберем гиперпараметры для биннинга переменных по WoE

In [ ]:
features = filtred_features

In [ ]:
cat_features = X_train[features].select_dtypes(include=['object', 'category']).columns.to_list()

In [ ]:
def gini_lr(values, target):
    """Джини через логрег"""
    values = np.reshape(values, (-1, 1))
    model = LR(random_state=RANDOM_SEED).fit(values, target)
    return round((2 * metrics.roc_auc_score(target, model.predict_proba(values)[:, 1]) - 1) * 100, 4)

In [ ]:
cv = StratifiedKFold()

In [ ]:
def objective_single_var(trial, X_train, y_train, variable, variable_type='numerical'):

    if variable_type == 'numerical':
        params = {
            "max_n_bins": trial.suggest_int("max_n_bins", 3, 10),
            "min_bin_size": trial.suggest_float("min_bin_size", 0.01, 0.2),
            "max_bin_size": trial.suggest_float("max_bin_size", 0.5, 1.0),
            "dtype": "numerical"
        }
    else:        params = {
            "max_n_bins": trial.suggest_int("max_n_bins", 2, min(20, len(X_train[variable].unique()))),
            "min_bin_size": trial.suggest_float("min_bin_size", 0.01, 0.3),
            "dtype": "categorical"
        }

    w = []

    for train_index, val_index in cv.split(X_train, y_train):
        binner = OptimalBinning(name=variable, **params)

        x_train_opt, x_valid_opt = X_train[features].iloc[train_index], X_train[features].iloc[val_index]
        y_train_opt, y_valid_opt = y_train.iloc[train_index], y_train.iloc[val_index]

        binner.fit(x_train_opt[variable], y_train_opt)
        x_train_opt_binned = binner.transform(x_train_opt[variable], metric="woe")
        x_valid_opt_binned = binner.transform(x_valid_opt[variable], metric="woe")

        # Оценка качества бининга через Gini
        w.append([
            gini_lr(x_train_opt_binned, y_train_opt),
            gini_lr(x_valid_opt_binned, y_valid_opt),
        ])

    res = list(np.mean(w, axis=0))

    return res[1]

In [ ]:
best_params_per_var = {}

In [ ]:
sampler = TPESampler(seed=RANDOM_SEED)

In [ ]:
for variable in tqdm(features):
    print(f"Переменная: {variable}, тип данных: {'categorical' if variable in cat_features else 'numerical'}")
    study = optuna.create_study(direction="maximize", sampler=sampler)
    if variable in cat_features:
        study.optimize(
            lambda trial: objective_single_var(trial, X_train, y_train, variable, 'categorical'),
            n_trials=20,
        )
    else:
        study.optimize(
            lambda trial: objective_single_var(trial, X_train, y_train, variable, 'numerical'),
            n_trials=20,
        )
    best_params_per_var[variable] = {
        "params": study.best_params,
        "gini": study.best_value,
    }

In [ ]:
# Вывод результатов
for var, params in best_params_per_var.items():
    print(f"Переменная: {var}")
    print(f"Лучшие параметры: {params['params']}")
    print(f"Gini: {params['gini']:.3f}\n")

## Закодируем переменные по WoE по полученным гиперпараметрам, оценим качество модели и подберем гиперпараметры модели

In [ ]:
# Закодируем переменные на основе отобранных значений гиперпараметров
X_train_binned = pd.DataFrame()
X_test_binned = pd.DataFrame()
X_oot_binned = pd.DataFrame()
X_oot_prom_binned = pd.DataFrame()

for feature, params in best_params_per_var.items():
    binning = OptimalBinning(
        name=feature,
        **params['params'],
        dtype='categorical' if feature in cat_features else 'numerical'
    )
    X_train_binned[feature] = binning.fit_transform(X_train[feature], y_train, metric="woe")
    X_test_binned[feature] = binning.transform(X_test[feature], metric="woe")
    X_oot_binned[feature] = binning.transform(X_oot[feature], metric="woe")
    X_oot_prom_binned[feature] = binning.transform(prom_oot[feature], metric="woe")

In [ ]:
model = LR(random_state=RANDOM_SEED).fit(X_train_binned, y_train)

In [ ]:
print(f"Gini train: {get_gini_lr(model, X_train_binned, y_train)}")
print(f"Gini test: {get_gini_lr(model, X_test_binned, y_test)}")
print(f"Gini oot: {get_gini_lr(model, X_oot_binned, y_oot)}")
print(f"Gini oot_prom: {get_gini_lr(model, X_oot_prom_binned, prom_oot[target])}")

## Подберем гиперпараметры модели

In [ ]:
cv = StratifiedKFold()

In [ ]:
def objective(trial):
    params = {
        "C": trial.sugg


est_float("C", 1e-5, 100, log=True),        "penalty": trial.suggest_categorical("penalty", ["l2", "none"]),

Шарнин Никита Сергеевич:
"solver": trial.suggest_categorical("solver", ["lbfgs", "saga"]),
        "max_iter": trial.suggest_int("max_iter", 100, 1000),
        "random_state": RANDOM_SEED
    }

    w = []

    for train_index, val_index in cv.split(X_train_binned, y_train):

        x_train_opt, x_valid_opt = X_train_binned[features].iloc[train_index], X_train_binned[features].iloc[val_index]
        y_train_opt, y_valid_opt = y_train.iloc[train_index], y_train.iloc[val_index]

        model = LR(**params).fit(x_train_opt, y_train_opt)

        w.append([
            round(2 * roc_auc_score(y_train_opt, model.predict_proba(x_train_opt)[:, 1]) - 1, 4),
            round(2 * roc_auc_score(y_valid_opt, model.predict_proba(x_valid_opt)[:, 1]) - 1, 4)
        ])

    res = list(np.mean(w, axis=0))

    trial.set_user_attr("gini_train", res[0])
    trial.set_user_attr("gini_test", res[1])

    return res[1]

In [ ]:
sampler = TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=30)

In [ ]:
optuna_report_df = study.trials_dataframe(
    attrs=("number", "value", "params", "user_attrs")
)

optuna_report_df = optuna_report_df.sort_values(by=['user_attrs_gini_test', 'user_attrs_gini_train'], ascending=False).reset_index()

best_lr_params = {'C': optuna_report_df.iloc[0]['params_C'],
                 'max_iter': optuna_report_df.iloc[0]['params_max_iter'],
                 'penalty': optuna_report_df.iloc[0]['params_penalty'],
                 'solver': optuna_report_df.iloc[0]['params_solver'],
                  "random_state": RANDOM_SEED
}

## Замерим качество полученной модели

In [ ]:
binning_tables = {}
splits = {}
binnings = {}

# Закодируем переменные на основе отобранных значений гиперпараметров
X_train_binned = pd.DataFrame()
X_test_binned = pd.DataFrame()
X_oot_binned = pd.DataFrame()

for feature, params in best_params_per_var.items():
    binning = OptimalBinning(
        name=feature,
        **params['params'],
        dtype='categorical' if feature in cat_features else 'numerical'
    )

    X_train_binned[feature] = binning.fit_transform(X_train[feature], y_train, metric="woe")
    X_test_binned[feature] = binning.transform(X_test[feature], metric="woe")
    X_oot_binned[feature] = binning.transform(X_oot[feature], metric="woe")
    X_oot_prom_binned[feature] = binning.transform(prom_oot[feature], metric="woe")

    # Сохраняем таблицу с информацией о разбиении переменных
    binning_table = binning.binning_table
    binning_tables[feature] = binning_table

    splits[feature] = binning.splits
    binnings[feature] = binning

In [ ]:
model = LR(**best_lr_params).fit(X_train_binned[features], y_train)

In [ ]:
print(f"Gini train: {get_gini_lr(model, X_train_binned, y_train)}")
print(f"Gini test: {get_gini_lr(model, X_test_binned, y_test)}")
print(f"Gini oot: {get_gini_lr(model, X_oot_binned, y_oot)}")
print(f"Gini oot_prom: {get_gini_lr(model, X_oot_prom_binned, prom_oot[target])}")

## Оценим качество модели на кроссвалидации

In [ ]:
# Кросс-валидация
tscv = TimeSeriesSplit()

ans = my_binnig_tqdm_cv(X_train_binned[features], y_train, tscv)
print(f"mean_train_cv_score: {ans['train']}, mean_test_cv_score: {ans['test']}")

## Сохраним результаты

In [ ]:
clf.save_model(os.path.join(MODELS_PATH, "logreg.cbm"))

In [ ]:
with open(os.path.join(MODELS_PATH, "params.json"), "w") as f:
  json.dump(clf.get_all_params(), f)